In [ ]:
# !pip install yt_dlp

In [2]:
import os
from tqdm import tqdm
import yt_dlp
import concurrent.futures

In [3]:
from datasets import load_dataset

# Login using e.g. `huggingface-cli login` to access this dataset
ds = load_dataset("FreedomIntelligence/TalkVid")

/home/danya/develop/students-materials/ML/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm

A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.2.6 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/home/danya/develop/students-materials/ML/.venv/lib/python3.12/site-packages/ipykernel_launcher.py", line 18, in <module>
    app.lau

In [4]:
ds

DatasetDict({
    test: Dataset({
        features: ['id', 'video-path', 'audio-path', 'height', 'width', 'fps', 'start-time', 'start-frame', 'end-time', 'end-frame', 'durations', 'dover_scores', 'cotracker_ratio', 'head_detail', 'info'],
        num_rows: 500
    })
})

In [5]:
ds['test'][0]

{'id': 'videovideo-hOQAsNg8aw-scene3-scene3',
 'video-path': 'TalkVid-bench/age/videos/videovideo-hOQAsNg8aw-scene3_scene3.mp4',
 'audio-path': 'TalkVid-bench/age/audios/videovideo-hOQAsNg8aw-scene3_scene3.m4a',
 'height': 1080,
 'width': 1920,
 'fps': 24,
 'start-time': 56.91973333333333,
 'start-frame': 1342,
 'end-time': 61.9614,
 'end-frame': 1706,
 'durations': '5.042s',
 'dover_scores': 7.88,
 'cotracker_ratio': 0.896938502788543,
 'head_detail': {'passed': True,
  'scores': {'avg_completeness': 100.0,
   'avg_movement': 97.84818459302187,
   'avg_orientation': 94.76435188709294,
   'avg_resolution': 327.0113193841628,
   'avg_rotation': 93.63263734580164,
   'min_completeness': 100.0,
   'min_movement': 95.92768885195255,
   'min_orientation': 92.89896301523412,
   'min_resolution': 270.7872404875579,
   'min_rotation': 82.97947249685893}},
 'info': {'Age Group': '60+',
  'Ethnicity': 'White',
  'Gender': 'Male',
  'Language': 'English',
  'Person ID': '',
  'Video Link': 'https

In [6]:
def download_single_video_optimized(item, output_path, resolution):
    """Оптимизированная загрузка одного видео"""
    video_url = item['info']['Video Link']
    dataset_video_id = item['id']
    
    ydl_opts = {
        'outtmpl': os.path.join(output_path, f'{dataset_video_id}.%(ext)s'),
        'format': f'best[height<={resolution}]/best',
        'merge_output_format': 'mp4',
        'concurrent_fragment_downloads': 8,
        'http_chunk_size': 8388608,  # 8MB
        'retries': 2,
        'fragment_retries': 2,
        'skip_unavailable_fragments': True,
        'noprogress': True,
        'quiet': True,
        'no_warnings': True,
        'postprocessors': [
            {
                'key': 'FFmpegVideoConvertor',
                'preferedformat': 'mp4',
            }
        ],
    }
    
    try:
        with yt_dlp.YoutubeDL(ydl_opts) as ydl:
            ydl.download([video_url])
        return dataset_video_id, True, None
    except Exception as e:
        return dataset_video_id, False, str(e)

def download_dataset_videos_ultimate(dataset, split='test', output_path="/home/danya/datasets/VidTalk/dataset_videos", resolution=720, max_workers=12):
    """Максимально оптимизированное параллельное скачивание"""
    if not os.path.exists(output_path):
        os.makedirs(output_path)
    
    successful_downloads = 0
    failed_downloads = []
    
    with concurrent.futures.ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = {
            executor.submit(download_single_video_optimized, item, output_path, resolution): item 
            for item in dataset[split]
        }
        
        for future in tqdm(concurrent.futures.as_completed(futures), total=len(futures), desc="Скачивание видео"):
            video_id, success, error = future.result()
            if success:
                successful_downloads += 1
            else:
                failed_downloads.append((video_id, error))
                print(f"Ошибка: {video_id} - {error}")
    
    print(f"Успешно скачано: {successful_downloads}")
    print(f"Не удалось скачать: {len(failed_downloads)}")
    
    return successful_downloads, failed_downloads

In [7]:
successful_downloads, failed_downloads = download_dataset_videos_ultimate(ds, max_workers=16)

Скачивание видео:   5%|█▌                              | 25/500 [02:18<33:39,  4.25s/it]

Ошибка: videovideoKvJ0zBpx3LY-scene200-scene3 - ERROR: [youtube] KvJ0zBpx3LY: Video unavailable. This video is no longer available because the YouTube account associated with this video has been terminated.


Скачивание видео:   5%|█▋                              | 27/500 [02:33<42:12,  5.35s/it]

Ошибка: videovideoLynxkVaKPeQ-scene1-scene23 - ERROR: [youtube] LynxkVaKPeQ: Private video. Sign in if you've been granted access to this video. Use --cookies-from-browser or --cookies for the authentication. See  https://github.com/yt-dlp/yt-dlp/wiki/FAQ#how-do-i-pass-cookies-to-yt-dlp  for how to manually pass cookies. Also see  https://github.com/yt-dlp/yt-dlp/wiki/Extractors#exporting-youtube-cookies  for tips on effectively exporting YouTube cookies


Скачивание видео:  12%|███▉                            | 62/500 [06:09<29:33,  4.05s/it]

...

Ошибка: videovideokmxqhUqjVP8-scene5-scene70 - ERROR: [youtube] kmxqhUqjVP8: Private video. Sign in if you've been granted access to this video. Use --cookies-from-browser or --cookies for the authentication. See  https://github.com/yt-dlp/yt-dlp/wiki/FAQ#how-do-i-pass-cookies-to-yt-dlp  for how to manually pass cookies. Also see  https://github.com/yt-dlp/yt-dlp/wiki/Extractors#exporting-youtube-cookies  for tips on effectively exporting YouTube cookies


Скачивание видео:  94%|█████████████████████████████  | 469/500 [36:36<01:34,  3.06s/it]

Ошибка: videovideotlLv1hdI2vs-scene21-scene6 - ERROR: [youtube] tlLv1hdI2vs: Private video. Sign in if you've been granted access to this video. Use --cookies-from-browser or --cookies for the authentication. See  https://github.com/yt-dlp/yt-dlp/wiki/FAQ#how-do-i-pass-cookies-to-yt-dlp  for how to manually pass cookies. Also see  https://github.com/yt-dlp/yt-dlp/wiki/Extractors#exporting-youtube-cookies  for tips on effectively exporting YouTube cookies


Скачивание видео: 100%|███████████████████████████████| 500/500 [38:51<00:00,  4.66s/it]

Успешно скачано: 479
Не удалось скачать: 21


In [8]:
successful_downloads
failed_downloads

[('videovideoKvJ0zBpx3LY-scene200-scene3',
  'ERROR: [youtube] KvJ0zBpx3LY: Video unavailable. This video is no longer available because the YouTube account associated with this video has been terminated.'),
 ('videovideoLynxkVaKPeQ-scene1-scene23',
  "ERROR: [youtube] LynxkVaKPeQ: Private video. Sign in if you've been granted access to this video. Use --cookies-from-browser or --cookies for the authentication. See  https://github.com/yt-dlp/yt-dlp/wiki/FAQ#how-do-i-pass-cookies-to-yt-dlp  for how to manually pass cookies. Also see  https://github.com/yt-dlp/yt-dlp/wiki/Extractors#exporting-youtube-cookies  for tips on effectively exporting YouTube cookies"),
 ('videovideoeI2V8Bd5X9s-scene6-scene1',
  "ERROR: [youtube] eI2V8Bd5X9s: Private video. Sign in if you've been granted access to this video. Use --cookies-from-browser or --cookies for the authentication. See  https://github.com/yt-dlp/yt-dlp/wiki/FAQ#how-do-i-pass-cookies-to-yt-dlp  for how to manually pass cookies. Also see  ht

In [ ]:
[('videovideoKvJ0zBpx3LY-scene200-scene3',
  'ERROR: [youtube] KvJ0zBpx3LY: Video unavailable. This video is no longer available because the YouTube account associated with this video has been terminated.'),
 ('videovideoLynxkVaKPeQ-scene1-scene23',
  "ERROR: [youtube] LynxkVaKPeQ: Private video. Sign in if you've been granted access to this video. Use --cookies-from-browser or --cookies for the authentication. See  https://github.com/yt-dlp/yt-dlp/wiki/FAQ#how-do-i-pass-cookies-to-yt-dlp  for how to manually pass cookies. Also see  https://github.com/yt-dlp/yt-dlp/wiki/Extractors#exporting-youtube-cookies  for tips on effectively exporting YouTube cookies"),
 ('videovideoeI2V8Bd5X9s-scene6-scene1',
  "ERROR: [youtube] eI2V8Bd5X9s: Private video. Sign in if you've been granted access to this video. Use --cookies-from-browser or --cookies for the authentication. See  https://github.com/yt-dlp/yt-dlp/wiki/FAQ#how-do-i-pass-cookies-to-yt-dlp  for how to manually pass cookies. Also see  https://github.com/yt-dlp/yt-dlp/wiki/Extractors#exporting-youtube-cookies  for tips on effectively exporting YouTube cookies"),
 ('videovideo_IAu7gYVZgNY-scene68-scene5',
  "ERROR: [youtube] B2zQ0oVZnGU: Private video. Sign in if you've been granted access to this video. Use --cookies-from-browser or --cookies for the authentication. See  https://github.com/yt-dlp/yt-dlp/wiki/FAQ#how-do-i-pass-cookies-to-yt-dlp  for how to manually pass cookies. Also see  https://github.com/yt-dlp/yt-dlp/wiki/Extractors#exporting-youtube-cookies  for tips on effectively exporting YouTube cookies"),
 ('videovideowhGh3Hyq1ew-scene13-scene1',
  'ERROR: [youtube] whGh3Hyq1ew: Video unavailable'),
 ('videovideotlLv1hdI2vs-scene19-scene10',
  "ERROR: [youtube] tlLv1hdI2vs: Private video. Sign in if you've been granted access to this video. Use --cookies-from-browser or --cookies for the authentication. See  https://github.com/yt-dlp/yt-dlp/wiki/FAQ#how-do-i-pass-cookies-to-yt-dlp  for how to manually pass cookies. Also see  https://github.com/yt-dlp/yt-dlp/wiki/Extractors#exporting-youtube-cookies  for tips on effectively exporting YouTube cookies"),
 ('videovideoC31IJoDcBhA-scene3-scene1',
  "ERROR: [youtube] C31IJoDcBhA: Private video. Sign in if you've been granted access to this video. Use --cookies-from-browser or --cookies for the authentication. See  https://github.com/yt-dlp/yt-dlp/wiki/FAQ#how-do-i-pass-cookies-to-yt-dlp  for how to manually pass cookies. Also see  https://github.com/yt-dlp/yt-dlp/wiki/Extractors#exporting-youtube-cookies  for tips on effectively exporting YouTube cookies"),
 ('videovideo3252_B_stWSZ7UJs-scene1-scene5',
  "ERROR: [youtube] B_stWSZ7UJs: Private video. Sign in if you've been granted access to this video. Use --cookies-from-browser or --cookies for the authentication. See  https://github.com/yt-dlp/yt-dlp/wiki/FAQ#how-do-i-pass-cookies-to-yt-dlp  for how to manually pass cookies. Also see  https://github.com/yt-dlp/yt-dlp/wiki/Extractors#exporting-youtube-cookies  for tips on effectively exporting YouTube cookies"),
 ('videovideouxHBm5Khcw8-scene2-scene20',
  "ERROR: [youtube] uxHBm5Khcw8: Private video. Sign in if you've been granted access to this video. Use --cookies-from-browser or --cookies for the authentication. See  https://github.com/yt-dlp/yt-dlp/wiki/FAQ#how-do-i-pass-cookies-to-yt-dlp  for how to manually pass cookies. Also see  https://github.com/yt-dlp/yt-dlp/wiki/Extractors#exporting-youtube-cookies  for tips on effectively exporting YouTube cookies"),
 ('videovideo2227_z2l0eqsFYDQ-scene3-scene121',
  "ERROR: [youtube] z2l0eqsFYDQ: Private video. Sign in if you've been granted access to this video. Use --cookies-from-browser or --cookies for the authentication. See  https://github.com/yt-dlp/yt-dlp/wiki/FAQ#how-do-i-pass-cookies-to-yt-dlp  for how to manually pass cookies. Also see  https://github.com/yt-dlp/yt-dlp/wiki/Extractors#exporting-youtube-cookies  for tips on effectively exporting YouTube cookies"),
 ('videovideo8593_Oq23eD8_zE0-scene20-scene18',
  "ERROR: Unable to download video: [Errno 2] No such file or directory: '/home/danya/datasets/VidTalk/dataset_videos/videovideo8593_Oq23eD8_zE0-scene20-scene18.mp4.part-Frag91'"),
 ('videovideoAe_brEqNahI-scene4-scene7',
  'ERROR: [youtube] Ae_brEqNahI: Video unavailable'),
 ('videovideo8515_Ii_8xSYg1lc-scene12-scene3',
  "ERROR: [youtube] c8-m8fBvZwU: Private video. Sign in if you've been granted access to this video. Use --cookies-from-browser or --cookies for the authentication. See  https://github.com/yt-dlp/yt-dlp/wiki/FAQ#how-do-i-pass-cookies-to-yt-dlp  for how to manually pass cookies. Also see  https://github.com/yt-dlp/yt-dlp/wiki/Extractors#exporting-youtube-cookies  for tips on effectively exporting YouTube cookies"),
 ('videovideoL0UUNeaG2FM-scene1-scene9',
  'ERROR: unable to download video data: HTTP Error 403: Forbidden'),
 ('videovideoUdqroxJAWZY-scene22-scene4',
  'ERROR: [youtube] UdqroxJAWZY: Video unavailable. This video is no longer available because the YouTube account associated with this video has been terminated.'),
 ('videovideo2iZjnutvjr8-scene9-scene37',
  'ERROR: [youtube] 2iZjnutvjr8: Video unavailable'),
 ('videovideoC31IJoDcBhA-scene14-scene3',
  "ERROR: [youtube] C31IJoDcBhA: Private video. Sign in if you've been granted access to this video. Use --cookies-from-browser or --cookies for the authentication. See  https://github.com/yt-dlp/yt-dlp/wiki/FAQ#how-do-i-pass-cookies-to-yt-dlp  for how to manually pass cookies. Also see  https://github.com/yt-dlp/yt-dlp/wiki/Extractors#exporting-youtube-cookies  for tips on effectively exporting YouTube cookies"),
 ('videovideo_YdhCCC0Hcz4-scene18-scene7',
  'ERROR: [youtube] YdhCCC0Hcz4: Video unavailable'),
 ('videovideo_vnIWg8ZLhXI-scene8-scene1',
  'ERROR: [youtube] vnIWg8ZLhXI: Video unavailable'),
 ('videovideokmxqhUqjVP8-scene5-scene70',
  "ERROR: [youtube] kmxqhUqjVP8: Private video. Sign in if you've been granted access to this video. Use --cookies-from-browser or --cookies for the authentication. See  https://github.com/yt-dlp/yt-dlp/wiki/FAQ#how-do-i-pass-cookies-to-yt-dlp  for how to manually pass cookies. Also see  https://github.com/yt-dlp/yt-dlp/wiki/Extractors#exporting-youtube-cookies  for tips on effectively exporting YouTube cookies"),
 ('videovideotlLv1hdI2vs-scene21-scene6',
  "ERROR: [youtube] tlLv1hdI2vs: Private video. Sign in if you've been granted access to this video. Use --cookies-from-browser or --cookies for the authentication. See  https://github.com/yt-dlp/yt-dlp/wiki/FAQ#how-do-i-pass-cookies-to-yt-dlp  for how to manually pass cookies. Also see  https://github.com/yt-dlp/yt-dlp/wiki/Extractors#exporting-youtube-cookies  for tips on effectively exporting YouTube cookies")]

In [9]:
# def play_movie(file_path):
#     """Plays an MP4 file using the system's default media player."""
#     os.system(f"xdg-open {ds['test'][0][file_path]}")

# # Example usage:
# play_movie(ds['test'][0]['video-path'])

In [10]:
import json
import pickle
from datetime import datetime

# Сохраняем successful_downloads
successful_data = {
    'metadata': {
        'total_successful': len(successful_downloads),
        'saved_at': datetime.now().isoformat(),
        'dataset_split': split
    },
    'successful_downloads': successful_downloads
}

with open('successful_downloads.json', 'w', encoding='utf-8') as f:
    json.dump(successful_data, f, indent=2, ensure_ascii=False)

print(f"✓ Успешные загрузки сохранены в successful_downloads.json")
print(f"  Количество: {len(successful_downloads)}")

# Сохраняем failed_downloads
failed_data = {
    'metadata': {
        'total_failed': len(failed_downloads),
        'saved_at': datetime.now().isoformat(),
        'dataset_split': split
    },
    'failed_downloads': failed_downloads
}

with open('failed_downloads.json', 'w', encoding='utf-8') as f:
    json.dump(failed_data, f, indent=2, ensure_ascii=False)

print(f"✓ Неудачные загрузки сохранены в failed_downloads.json")
print(f"  Количество: {len(failed_downloads)}")

# Дополнительно: сохраняем в pickle для быстрой загрузки
with open('successful_downloads.pkl', 'wb') as f:
    pickle.dump(successful_downloads, f)

with open('failed_downloads.pkl', 'wb') as f:
    pickle.dump(failed_downloads, f)

print("✓ Данные также сохранены в pickle формате")

TypeError: object of type 'int' has no len()